# LazyRegressor screening + top-3 tuning — `route` dataset

Task T4.route.lazy. Screens the CURATED regressor list from `02_lazy.md` with
`LazyRegressor` on a group-disjoint inner split of train, tunes the top-3 by
validation R² with Optuna (grouped CV over whole training days, objective = mean MAE), then refits
each on the full train split and scores test **once**.

Dataset: `route` (Green route, all buses/days; row/feature counts printed
below — never hard-code them, the dataset build has changed under us
before). Target: `delay_s`.

Split (v2): whole **service days** held out as test (fixes v1's hour-block
leakage, where labels ran up to 45 min across hour boundaries). Tuning
`cv_group` = the service date for `route` (2 train days, so tuning uses `GroupKFold(2)`: train on one day, validate on the other). Fixed by the dataset build
(T2), loaded via `mc_common`.

In [1]:
import sys, time, inspect
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

import numpy as np
import optuna
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (AdaBoostRegressor, BaggingRegressor, ExtraTreesRegressor,
                               GradientBoostingRegressor, HistGradientBoostingRegressor,
                               RandomForestRegressor)
from sklearn.linear_model import (BayesianRidge, ElasticNet, HuberRegressor, Lars, Lasso,
                                   LassoLars, LinearRegression, OrthogonalMatchingPursuit,
                                   PassiveAggressiveRegressor, Ridge, SGDRegressor)
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVR
from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from lightgbm import LGBMRegressor
from lazypredict.Supervised import LazyRegressor

from mc_common import SMOKE, THREADS, budget, group_val_split, load_model_ready, save_result, set_seed

set_seed(42)
optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f"SMOKE={SMOKE}  THREADS={THREADS}")

SMOKE=False  THREADS=2


## L1 — load the model-ready data
`load_model_ready("route")` reads `datasets/route_model_ready.parquet` +
schema and returns the fixed train/test split (features, target `delay_s`,
`cv_group` for train). Shapes printed below.

In [2]:
d = load_model_ready("route")
print(f"train: {d.X_train.shape}   test: {d.X_test.shape}   features: {len(d.features)}")
print(f"unique cv_group in train: {len(np.unique(d.g_train))}")

train: (43811, 43)   test: (16998, 43)   features: 43
unique cv_group in train: 2


## L2 — group-disjoint inner split + subsample

`group_val_split` carves a validation slice out of TRAIN only (test is never
touched here). Both pieces are then subsampled for the screening step so
`LazyRegressor` runs fast: inner-train ≤ `budget(15000, 2000)` rows, val ≤
`budget(6000, 1000)` rows.

In [3]:
X_inner, X_val, y_inner, y_val, g_inner = group_val_split(d.X_train, d.y_train, d.g_train, frac=0.2, seed=42)


def subsample(X, y, n, seed=42):
    """Random row subsample (no replacement), capped at n rows."""
    if len(X) <= n:
        return X, y
    idx = X.sample(n=n, random_state=seed).index
    return X.loc[idx], y.loc[idx]


X_screen, y_screen = subsample(X_inner, y_inner, budget(15000, 2000))
X_val_screen, y_val_screen = subsample(X_val, y_val, budget(6000, 1000))
print(f"screen train: {X_screen.shape}   screen val: {X_val_screen.shape}")

screen train: (15000, 43)   screen val: (6000, 43)


## L3 — LazyRegressor screening

CURATED list (24 models) per `02_lazy.md`. Excluded and why:
- **XGBRegressor** — has its own notebook (`xgboost_route.ipynb`).
- **SVR / NuSVR / KernelRidge / GaussianProcessRegressor** — O(n²)–O(n³) fit, too slow at this row count.
- **QuantileRegressor / TheilSenRegressor / RANSACRegressor** — slow or unstable at this scale.
- **PoissonRegressor / GammaRegressor / TweedieRegressor** — need a strictly positive target; `delay_s` can be negative.

`LazyRegressor` builds its own preprocessing pipeline per model internally
(imputation/encoding); our features are already fully numeric so this is a
pass-through. It is used for screening only — final models are refit by hand
in L6.

In [4]:
CURATED = [
    LinearRegression, Ridge, Lasso, ElasticNet, Lars, LassoLars,
    OrthogonalMatchingPursuit, BayesianRidge, HuberRegressor, SGDRegressor,
    PassiveAggressiveRegressor, LinearSVR, KNeighborsRegressor,
    DecisionTreeRegressor, ExtraTreeRegressor, RandomForestRegressor,
    ExtraTreesRegressor, BaggingRegressor, GradientBoostingRegressor,
    HistGradientBoostingRegressor, AdaBoostRegressor, MLPRegressor,
    LGBMRegressor, DummyRegressor,
]
MODEL_CLASSES = {cls.__name__: cls for cls in CURATED}

lazy_reg = LazyRegressor(verbose=0, ignore_warnings=True, random_state=42, regressors=CURATED)
# this lazypredict build's fit() always returns (scores, predictions_df) regardless
# of the `predictions` flag -- the second element is unused here.
leaderboard, _ = lazy_reg.fit(X_screen, X_val_screen, y_screen, y_val_screen)
leaderboard

,Adjusted R-Squared,R-Squared,RMSE,Time Taken
Model,,,,
ExtraTreesRegressor,3.021082e-01,3.071106e-01,2.723821e+02,11.968832
AdaBoostRegressor,2.563139e-01,2.616445e-01,2.811767e+02,2.188838
LGBMRegressor,2.394630e-01,2.449144e-01,2.843444e+02,0.261052
GradientBoostingRegressor,2.307202e-01,2.362343e-01,2.859741e+02,4.891963
HistGradientBoostingRegressor,2.222750e-01,2.278496e-01,2.875396e+02,0.606924
RandomForestRegressor,1.916343e-01,1.974285e-01,2.931491e+02,13.974111
KNeighborsRegressor,1.902235e-01,1.960278e-01,2.934048e+02,0.385282
BaggingRegressor,1.637620e-01,1.697561e-01,2.981601e+02,1.449136
DecisionTreeRegressor,8.253506e-02,8.911132e-02,3.123053e+02,0.305852


## L4 — pick the top 3

Ranked by validation **R-Squared** (descending), `DummyRegressor` excluded
(it is the floor, not a candidate).

In [5]:
ranked = leaderboard.drop(index="DummyRegressor", errors="ignore").sort_values("R-Squared", ascending=False)
top3 = list(ranked.index[:3])
print("Top 3 by screening val R2:", top3)
ranked.head(10)

Top 3 by screening val R2: ['ExtraTreesRegressor', 'AdaBoostRegressor', 'LGBMRegressor']


,Adjusted R-Squared,R-Squared,RMSE,Time Taken
Model,,,,
ExtraTreesRegressor,0.302108,0.307111,272.382124,11.968832
AdaBoostRegressor,0.256314,0.261645,281.176744,2.188838
LGBMRegressor,0.239463,0.244914,284.344446,0.261052
GradientBoostingRegressor,0.230720,0.236234,285.974116,4.891963
HistGradientBoostingRegressor,0.222275,0.227850,287.539552,0.606924
RandomForestRegressor,0.191634,0.197429,293.149063,13.974111
KNeighborsRegressor,0.190223,0.196028,293.404759,0.385282
BaggingRegressor,0.163762,0.169756,298.160087,1.449136
DecisionTreeRegressor,0.082535,0.089111,312.305262,0.305852


## L5 — Optuna tuning of the top 3

Search spaces cover **every** CURATED model (so whichever 3 win screening
have a space ready). Linear/KNN/MLP/SVR models run inside
`Pipeline([StandardScaler, model])`; tree/boosting models don't need scaling.

Objective = mean MAE (seconds) over `GroupKFold(min(3, #train days))` (= 2 here) on a train subsample
(≤ `budget(30000, 3000)` rows, groups = `cv_group`). `n_trials=budget(40, 3)`,
`timeout=budget(900, 30)`s per model (v2: was 600), `TPESampler(seed=42)`,
`n_jobs=1` (sklearn/lightgbm parallelism uses `THREADS` instead — see
`build_model`).

A few "iteration count" hyperparameters (`n_estimators`, `max_iter`, ...) have
their upper bound scaled by `budget()` too, so a smoke run doesn't spend its
5-minute budget fitting a 2000-tree model on the full train set at L6; the
full run keeps the exact ranges from `02_lazy.md`.

Each trial stashes the exact sklearn kwargs it used via
`trial.set_user_attr("model_params", params)`; the winning trial's kwargs are
read back from `study.best_trial.user_attrs` for the L6 refit rather than
reconstructed from `study.best_params`, so a future conditional search space
can't silently drift from what actually got fit.

`HistGradientBoostingRegressor` forces `early_stopping=False` (v2) in every
trial and in the L6 refit, and tunes `max_iter` directly — its own default
early stopping holds out a random, non-grouped 10% of whatever it's given,
which would leak across `cv_group`.

In [6]:
SCALE_NEEDED = {
    "LinearRegression", "Ridge", "Lasso", "ElasticNet", "Lars", "LassoLars",
    "OrthogonalMatchingPursuit", "BayesianRidge", "HuberRegressor", "SGDRegressor",
    "PassiveAggressiveRegressor", "LinearSVR", "KNeighborsRegressor", "MLPRegressor",
}
# Fixed (non-tuned) kwargs some models always need; kept out of the Optuna search
# space so study.best_params stays a clean set of *tuned* hyperparameters.
FIXED_EXTRA = {
    "MLPRegressor": {"early_stopping": True},
    "LGBMRegressor": {"subsample_freq": 1, "verbose": -1},
    # v2: HistGB's own early stopping holds out a random, non-grouped 10% --
    # off in both the search space and the final refit; max_iter is tuned instead.
    "HistGradientBoostingRegressor": {"early_stopping": False},
}
N_FEATS = len(d.features)


def suggest_params(trial, name):
    """Optuna search space per CURATED model (protocol 02_lazy.md L5).
    max_depth/max_features use a discrete grid (incl. None/'sqrt') instead of a
    branching conditional space, so study.best_params is directly usable as
    model kwargs with no replay step.
    """
    if name == "LinearRegression":
        return {"fit_intercept": trial.suggest_categorical("fit_intercept", [True, False])}
    if name == "Ridge":
        return {"alpha": trial.suggest_float("alpha", 1e-3, 100, log=True)}
    if name == "Lasso":
        return {"alpha": trial.suggest_float("alpha", 1e-4, 10, log=True)}
    if name == "ElasticNet":
        return {"alpha": trial.suggest_float("alpha", 1e-4, 10, log=True),
                "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0)}
    if name == "Lars":
        return {"n_nonzero_coefs": trial.suggest_int("n_nonzero_coefs", 1, N_FEATS)}
    if name == "LassoLars":
        return {"alpha": trial.suggest_float("alpha", 1e-4, 10, log=True)}
    if name == "OrthogonalMatchingPursuit":
        return {"n_nonzero_coefs": trial.suggest_int("n_nonzero_coefs", 1, N_FEATS)}
    if name == "BayesianRidge":
        return {"alpha_1": trial.suggest_float("alpha_1", 1e-8, 1e-2, log=True),
                "lambda_1": trial.suggest_float("lambda_1", 1e-8, 1e-2, log=True)}
    if name == "HuberRegressor":
        return {"epsilon": trial.suggest_float("epsilon", 1.05, 5.0),
                "alpha": trial.suggest_float("alpha", 1e-6, 1e-1, log=True)}
    if name == "SGDRegressor":
        return {"alpha": trial.suggest_float("alpha", 1e-6, 1e-1, log=True),
                "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
                "penalty": trial.suggest_categorical("penalty", ["l2", "l1", "elasticnet"])}
    if name == "PassiveAggressiveRegressor":
        return {"C": trial.suggest_float("C", 1e-3, 10, log=True),
                "epsilon": trial.suggest_float("epsilon", 0.01, 1.0)}
    if name == "LinearSVR":
        return {"C": trial.suggest_float("C", 1e-3, 10, log=True),
                "epsilon": trial.suggest_float("epsilon", 0.0, 1.0)}
    if name == "KNeighborsRegressor":
        return {"n_neighbors": trial.suggest_int("n_neighbors", 3, 100),
                "weights": trial.suggest_categorical("weights", ["uniform", "distance"]),
                "p": trial.suggest_categorical("p", [1, 2])}
    if name in ("DecisionTreeRegressor", "ExtraTreeRegressor"):
        return {"max_depth": trial.suggest_categorical("max_depth", [None, 3, 5, 8, 12, 16, 20, 25, 30]),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50)}
    if name in ("RandomForestRegressor", "ExtraTreesRegressor"):
        return {"n_estimators": trial.suggest_int("n_estimators", 100, budget(600, 150)),
                "max_depth": trial.suggest_categorical("max_depth", [None, 5, 10, 15, 20, 25, 30, 35, 40]),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
                "max_features": trial.suggest_categorical("max_features", ["sqrt", 0.3, 0.5, 0.7, 0.85, 1.0])}
    if name == "BaggingRegressor":
        return {"n_estimators": trial.suggest_int("n_estimators", 10, budget(200, 40)),
                "max_samples": trial.suggest_float("max_samples", 0.3, 1.0),
                "max_features": trial.suggest_float("max_features", 0.3, 1.0)}
    if name == "GradientBoostingRegressor":
        return {"n_estimators": trial.suggest_int("n_estimators", 100, budget(800, 150)),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "max_depth": trial.suggest_int("max_depth", 2, 8),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50)}
    if name == "HistGradientBoostingRegressor":
        return {"learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "max_iter": trial.suggest_int("max_iter", 100, budget(1000, 150)),
                "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 255),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 200),
                "l2_regularization": trial.suggest_float("l2_regularization", 0.0, 10.0)}
    if name == "AdaBoostRegressor":
        return {"n_estimators": trial.suggest_int("n_estimators", 50, budget(500, 100)),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 2.0, log=True),
                "loss": trial.suggest_categorical("loss", ["linear", "square", "exponential"])}
    if name == "MLPRegressor":
        return {"hidden_layer_sizes": trial.suggest_categorical(
                    "hidden_layer_sizes", [(64,), (128, 64), (256, 128), (128, 128, 64)]),
                "alpha": trial.suggest_float("alpha", 1e-6, 1e-2, log=True),
                "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
                "max_iter": trial.suggest_int("max_iter", 100, budget(300, 80))}
    if name == "LGBMRegressor":
        return {"n_estimators": trial.suggest_int("n_estimators", 100, budget(2000, 300)),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 15, 255),
                "min_child_samples": trial.suggest_int("min_child_samples", 5, 200),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10, log=True)}
    if name == "DummyRegressor":
        return {}
    raise ValueError(f"no search space for {name!r}")


def build_model(name, params):
    """Tuned params + fixed extras -> a fitted-ready estimator (or Pipeline w/ scaling)."""
    cls = MODEL_CLASSES[name]
    kwargs = {**FIXED_EXTRA.get(name, {}), **params}
    sig = inspect.signature(cls.__init__).parameters
    if "random_state" in sig:
        kwargs["random_state"] = 42
    if "n_jobs" in sig:
        kwargs["n_jobs"] = THREADS
    est = cls(**kwargs)
    return Pipeline([("scaler", StandardScaler()), ("model", est)]) if name in SCALE_NEEDED else est


def make_objective(name, X, y, groups):
    # route has 2 train days and tunes with whole-day groups -> at most 2 folds
    gkf = GroupKFold(n_splits=min(3, len(np.unique(groups))))
    def objective(trial):
        params = suggest_params(trial, name)
        # stash the exact tuned kwargs on the trial so L5's final refit reads
        # them back from the winning trial instead of replaying study.best_params
        trial.set_user_attr("model_params", params)
        maes = []
        for tr_idx, va_idx in gkf.split(X, y, groups):
            model = build_model(name, params)
            model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
            pred = model.predict(X.iloc[va_idx])
            maes.append(mean_absolute_error(y.iloc[va_idx], pred))
        return float(np.mean(maes))
    return objective

In [7]:
# Tuning subsample: drawn from the FULL train split (not the L2 inner split),
# groups = d.g_train, per protocol rule "Tuning CV = GroupKFold(3) on
# (sub-sampled) train with d.g_train groups."
tune_n = min(budget(30000, 3000), len(d.X_train))
tune_idx = d.X_train.sample(n=tune_n, random_state=42).index
X_tune = d.X_train.loc[tune_idx].reset_index(drop=True)
y_tune = d.y_train.loc[tune_idx].reset_index(drop=True)
g_tune = d.g_train[tune_idx.to_numpy()]  # d.X_train has a plain RangeIndex, so tune_idx doubles as positions
print(f"tuning subsample: {X_tune.shape}, {len(np.unique(g_tune))} groups")

studies, best_params = {}, {}
for name in top3:
    t0 = time.time()
    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(make_objective(name, X_tune, y_tune, g_tune),
                    n_trials=budget(40, 3), timeout=budget(900, 30), n_jobs=1)
    studies[name] = study
    best_params[name] = study.best_trial.user_attrs["model_params"]
    print(f"[{name}] best CV MAE={study.best_value:.1f}s  "
          f"({len(study.trials)} trials, {time.time() - t0:.1f}s)  params={best_params[name]}")

tuning subsample: (30000, 43), 2 groups


[ExtraTreesRegressor] best CV MAE=192.2s  (40 trials, 642.6s)  params={'n_estimators': 204, 'max_depth': 40, 'min_samples_leaf': 6, 'max_features': 0.5}


[AdaBoostRegressor] best CV MAE=202.6s  (40 trials, 836.3s)  params={'n_estimators': 106, 'learning_rate': 0.018516142991749522, 'loss': 'square'}


[LGBMRegressor] best CV MAE=199.9s  (40 trials, 70.1s)  params={'n_estimators': 179, 'learning_rate': 0.01443997778790209, 'num_leaves': 23, 'min_child_samples': 97, 'subsample': 0.5252518649470443, 'colsample_bytree': 0.6246289782867387, 'reg_alpha': 1.1787911086026723, 'reg_lambda': 7.994196869636166e-05}


## L6 — final fit on full train, predict test once

Each tuned model is refit on the **entire** train split (not the screening or
tuning subsamples) and scored on test exactly once via `save_result`, which
also writes `results/preds/route__lazy__<Model>.csv.gz`.

In [8]:
results = []
for rank, name in enumerate(top3, start=1):
    t0 = time.time()
    model = build_model(name, best_params[name])
    model.fit(d.X_train, d.y_train)
    fit_s = time.time() - t0
    y_pred = model.predict(d.X_test)
    rec = save_result(
        "route", "lazy", name, d.y_test, y_pred,
        params={**FIXED_EXTRA.get(name, {}), **best_params[name]},
        cv_mae=studies[name].best_value,
        extra={
            "screen_rank": rank,
            "screen_val_r2": float(ranked.loc[name, "R-Squared"]),
            "leaderboard_top10": ranked.head(10).reset_index().to_dict(orient="records"),
            "n_trials_complete": len(studies[name].trials),
            "fit_seconds": fit_s,
        },
    )
    results.append({**rec, "fit_seconds": fit_s})

[route__lazy__ExtraTreesRegressor] test R2=0.4568  MAE=135.6s  RMSE=179.3s  (n=16998)


[route__lazy__AdaBoostRegressor] test R2=0.3595  MAE=157.7s  RMSE=194.7s  (n=16998)


[route__lazy__LGBMRegressor] test R2=0.5053  MAE=139.9s  RMSE=171.1s  (n=16998)


## L7 — summary

Test-set R², MAE, RMSE and fit time for the 3 tuned models.

In [9]:
import pandas as pd

summary = pd.DataFrame([
    {"model": r["model"], "screen_rank": r["extra"]["screen_rank"],
     "test_r2": r["r2"], "test_mae_s": r["mae"], "test_rmse_s": r["rmse"],
     "cv_mae_s": r["cv_mae"], "fit_seconds": round(r["fit_seconds"], 1), "n_test": r["n_test"]}
    for r in results
]).sort_values("test_r2", ascending=False).reset_index(drop=True)
summary

,model,screen_rank,test_r2,test_mae_s,test_rmse_s,cv_mae_s,fit_seconds,n_test
0,LGBMRegressor,3,0.505304,139.873603,171.086330,199.920464,0.4,16998
1,ExtraTreesRegressor,1,0.456836,135.578122,179.271527,192.240486,8.2,16998
2,AdaBoostRegressor,2,0.359535,157.694925,194.667538,202.573097,13.6,16998
